Este notebook toma los EPT existentes para cada vendor-dow-block y los ajusta según las instrucciones vigentes de `to_adjust_now`.

La hoja de input se mantiene como una tabla de instrucciones. El detalle estandarizado por vendor se escribe por separado en `results_export`.


INPUT:
* `to_adjust_now`: `vendor_code` + `ept_new`; opcionales `franchise_id`, `wave` y `flag`.
* También acepta los aliases `Vendor code` y `EPT min` cuando no existen los nombres estándar.
* `EPT actuales`: resultados de la query por vendor-dow-block.

OUTPUT:
* CSV TES en Drive.
* `results_export`: una fila por vendor efectivamente exportado, con origen, alcance y detalle del ajuste.
* `last_executed_at` se actualiza en `to_adjust_now` solo al finalizar correctamente.


In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass


## Data Load

### instrucciones: `to_adjust_now`


In [2]:
reductions_only = False
required_diff_mins = 3

#### waves calculados

In [3]:
# # waves calculator
# file_name = 'new_preps_mod.csv'

# from google.colab import drive
# drive.mount('/content/drive')
# file_path_main = '/content/drive/MyDrive/lower ept y awt/ept reductions input data/'
# file_path = file_path_main + file_name
# new_preps = pd.read_csv(file_path)
# new_preps.head(3)

#### input unificado: waves y ajustes particulares


In [4]:
import re
import numpy as np
import pandas as pd
import gspread
import google.auth

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = "peya-chile"
QUOTA_PROJECT_ID = "dhub-data-commune"
GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

credentials, _ = google.auth.default(
    scopes=GOOGLE_SCOPES,
    quota_project_id=QUOTA_PROJECT_ID,
)

gc = gspread.authorize(credentials)

sheet_id = "1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE"
INPUT_SHEET_NAME = "to_adjust_now"
RESULTS_SHEET_NAME = "results_export"

spreadsheet = gc.open_by_key(sheet_id)
input_worksheet = spreadsheet.worksheet(INPUT_SHEET_NAME)
input_values = input_worksheet.get_all_values()

if not input_values or not input_values[0]:
    raise ValueError(f"La hoja '{INPUT_SHEET_NAME}' está vacía.")

input_headers_original = [str(column).strip() for column in input_values[0]]

if len(input_headers_original) != len(set(input_headers_original)):
    duplicates = pd.Series(input_headers_original).value_counts()
    duplicates = duplicates[duplicates > 1].index.tolist()
    raise ValueError(
        "Hay encabezados repetidos en to_adjust_now: "
        + ", ".join(map(str, duplicates))
    )

input_table = pd.DataFrame(
    input_values[1:],
    columns=input_headers_original
)
input_table["_input_row_number"] = range(2, len(input_table) + 2)

# Ignorar filas completamente vacías, sin considerar timestamps anteriores.
content_columns = [
    column for column in input_headers_original
    if column != "last_executed_at"
]

if content_columns:
    has_content = input_table[content_columns].apply(
        lambda row: row.astype("string").str.strip().ne("").any(),
        axis=1
    )
    input_table = input_table.loc[has_content].copy()

if input_table.empty:
    raise ValueError(f"La hoja '{INPUT_SHEET_NAME}' no tiene instrucciones.")


def normalized_header(value):
    return re.sub(r"[\s_-]+", " ", str(value).strip()).casefold()


def use_column_alias(df, target, aliases):
    """Usa un alias solo si falta la columna estándar; si ambas existen, completa vacíos."""
    lookup = {normalized_header(column): column for column in df.columns}
    alias_column = next(
        (
            lookup[normalized_header(alias)]
            for alias in aliases
            if normalized_header(alias) in lookup
            and lookup[normalized_header(alias)] != target
        ),
        None
    )

    if target not in df.columns:
        if alias_column is None:
            return df
        return df.rename(columns={alias_column: target})

    if alias_column is not None:
        target_blank = (
            df[target].astype("string").str.strip().isin(["", "nan", "<NA>"])
            | df[target].isna()
        )
        df.loc[target_blank, target] = df.loc[target_blank, alias_column]

    return df


column_aliases = {
    "vendor_code": ["Vendor code", "Vendor Code", "vendor code"],
    "ept_new": ["EPT min", "EPT Min", "ept min"],
    "franchise_id": ["Franchise ID", "Franchise id", "franchise id"],
    "wave": ["Wave"],
    "flag": ["Flag"],
}

for standard_name, aliases in column_aliases.items():
    input_table = use_column_alias(input_table, standard_name, aliases)

if "ept_new" not in input_table.columns:
    raise ValueError(
        "Falta la columna ept_new. También se acepta el alias 'EPT min'."
    )

# vendor_code puede faltar si todas las instrucciones vienen por franchise_id.
for optional_column in [
    "vendor_code", "franchise_id", "wave", "flag", "last_executed_at"
]:
    if optional_column not in input_table.columns:
        input_table[optional_column] = pd.NA

new_preps = input_table.copy()
input_rows_to_mark = input_table["_input_row_number"].astype(int).tolist()

execution_timestamp = pd.Timestamp.now(tz="America/Santiago").floor("s")
executed_at = execution_timestamp.strftime("%Y-%m-%d %H:%M:%S")
execution_id = execution_timestamp.strftime("%Y%m%d_%H%M%S")

print(
    f"Instrucciones leídas: {len(new_preps):,} | "
    f"Ejecución: {executed_at} America/Santiago"
)

new_preps.head()


Instrucciones leídas: 11 | Ejecución: 2026-09-10 17:33:34 America/Santiago


,wave,flag,franchise_id,vendor_code,ept_new,last_executed_at,_input_row_number
0,,reajuste regional,,291016,25,,2
1,,reajuste regional,,133509,25,,3
2,,reajuste regional,,341835,25,,4
3,,reajuste regional,,535773,25,,5
4,,reajuste regional,,133603,25,,6


In [5]:
def normalize_id(series):
    normalized = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    return normalized.mask(
        normalized.isna()
        | normalized.str.lower().isin(["", "nan", "none", "<na>"])
    )


new_preps["vendor_code"] = normalize_id(new_preps["vendor_code"])
new_preps["franchise_id"] = normalize_id(new_preps["franchise_id"])
new_preps["ept_new"] = pd.to_numeric(new_preps["ept_new"], errors="coerce")

for column in ["wave", "flag"]:
    new_preps[column] = (
        new_preps[column]
        .astype("string")
        .str.strip()
        .mask(lambda values: values.isna() | values.eq(""))
    )

rows_without_target = new_preps[
    new_preps["vendor_code"].isna()
    & new_preps["franchise_id"].isna()
]

if not rows_without_target.empty:
    raise ValueError(
        "Hay filas en to_adjust_now sin vendor_code ni franchise_id: "
        f"{rows_without_target['_input_row_number'].astype(int).tolist()}"
    )

rows_without_ept = new_preps[new_preps["ept_new"].isna()]

if not rows_without_ept.empty:
    raise ValueError(
        "Hay filas en to_adjust_now con ept_new vacío o no numérico: "
        f"{rows_without_ept['_input_row_number'].astype(int).tolist()}"
    )

# Si existe vendor_code, la instrucción es individual aunque también venga
# franchise_id. Una fila sin vendor_code se interpreta como franquicia completa.
new_preps["adjustment_scope"] = np.where(
    new_preps["vendor_code"].notna(),
    "vendor",
    "franchise"
)
new_preps["instruction_key"] = np.where(
    new_preps["adjustment_scope"].eq("vendor"),
    "vendor:" + new_preps["vendor_code"].astype("string"),
    "franchise:" + new_preps["franchise_id"].astype("string")
)

# No ocultar conflictos eligiendo silenciosamente el menor EPT.
conflicts = (
    new_preps.groupby("instruction_key")["ept_new"]
    .nunique(dropna=False)
)
conflicts = conflicts[conflicts > 1]

if not conflicts.empty:
    raise ValueError(
        "Hay instrucciones repetidas con distintos ept_new: "
        + ", ".join(conflicts.index.tolist())
    )

# Para duplicados idénticos se conserva la última fila de la hoja, incluyendo
# sus metadatos wave y flag. Las franquicias ya no colapsan por vendor_code vacío.
new_preps = (
    new_preps
    .sort_values("_input_row_number")
    .drop_duplicates("instruction_key", keep="last")
    .reset_index(drop=True)
)

print(
    f"Reglas válidas: {len(new_preps):,} | "
    f"vendors: {(new_preps['adjustment_scope'] == 'vendor').sum():,} | "
    f"franquicias: {(new_preps['adjustment_scope'] == 'franchise').sum():,}"
)

new_preps.head()


Reglas válidas: 11 | vendors: 11 | franquicias: 0


,wave,flag,franchise_id,vendor_code,ept_new,last_executed_at,_input_row_number,adjustment_scope,instruction_key
0,<NA>,reajuste regional,<NA>,291016,25,,2,vendor,vendor:291016
1,<NA>,reajuste regional,<NA>,133509,25,,3,vendor,vendor:133509
2,<NA>,reajuste regional,<NA>,341835,25,,4,vendor,vendor:341835
3,<NA>,reajuste regional,<NA>,535773,25,,5,vendor,vendor:535773
4,<NA>,reajuste regional,<NA>,133603,25,,6,vendor,vendor:133603


#### generate query

In [6]:
from datetime import timedelta
import pandas as pd

# ============================================================
# 1. IDs únicos para limitar la query
# ============================================================

vendor_codes = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("vendor"),
        "vendor_code"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)

franchise_ids = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("franchise"),
        "franchise_id"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)


def sql_string_list(values):
    return ",\n      ".join(
        f"'{value.replace(chr(39), chr(39) * 2)}'"
        for value in values
    )


target_conditions = []

if vendor_codes:
    target_conditions.append(
        "CAST(p.partner_id AS STRING) IN (\n"
        f"      {sql_string_list(vendor_codes)}\n"
        "    )"
    )

if franchise_ids:
    target_conditions.append(
        "CAST(p.franchise.franchise_id AS STRING) IN (\n"
        f"      {sql_string_list(franchise_ids)}\n"
        "    )"
    )

if not target_conditions:
    raise ValueError(
        "to_adjust_now no contiene vendor_code ni franchise_id válidos."
    )

target_filter_sql = "\n    OR ".join(target_conditions)

# ============================================================
# 2. Últimos 7 días completos terminando ayer, hora de Chile
# ============================================================

today = pd.Timestamp.now(tz="America/Santiago").date()
start_date = today - timedelta(days=7)
end_date = today - timedelta(days=1)


# ============================================================
# 3. Query
# ============================================================

query = f"""
WITH vendor_verticals AS (
  SELECT
    vendor_code,
    ARRAY_AGG(
      vertical_type IGNORE NULLS
      ORDER BY vertical_type
      LIMIT 1
    )[SAFE_OFFSET(0)] AS vertical_type

  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendors`

  WHERE entity_id = 'PY_CL'
    AND vendor_code IS NOT NULL

  GROUP BY vendor_code
),

hdm AS (
  SELECT
    SAFE_CAST(order_code AS INT64) AS order_code,

    high_demand_mode.is_hd_order AS hd_order,
    SAFE_CAST(
      high_demand_mode.minutes_added AS FLOAT64
    ) AS minutes_added,
    SAFE_CAST(
      high_demand_mode.duration AS FLOAT64
    ) AS duration,

    high_demand_mode.enabled_at,
    high_demand_mode.disabled_at

  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders`

  WHERE created_date BETWEEN DATE '{start_date}'
                         AND DATE '{end_date}'
    AND country_code = 'cl'
    AND high_demand_mode.is_hd_order IS TRUE
    AND requested_pickup_at IS NULL
),

target_stores AS (
  SELECT DISTINCT
    CAST(p.partner_id AS STRING) AS vendor_code,
    p.partner_name AS store_name,
    v.vertical_type,

    CAST(
      p.franchise.franchise_id AS STRING
    ) AS franchise_id,

    COALESCE(
      CAST(p.franchise.franchise_id AS STRING),
      CONCAT(
        'VENDOR_',
        CAST(p.partner_id AS STRING)
      )
    ) AS franchise_group_id,

    p.franchise.franchise_name AS franchise_name

  FROM `peya-bi-tools-pro.il_core.dim_partner` p

  LEFT JOIN vendor_verticals v
    ON CAST(p.partner_id AS STRING) = v.vendor_code

  WHERE p.is_online = TRUE
    AND (
      {target_filter_sql}
    )
),

base AS (
  SELECT
    l.peya_order_id,
    l.vendor.vendor_code AS vendor_code,

    COALESCE(
      p.store_name,
      l.vendor.name
    ) AS store_name,

    l.city.city_name AS city_name,

    p.franchise_id,
    p.franchise_group_id,
    p.franchise_name,

    l.food_is_ready_at,
    COALESCE(
      p.vertical_type,
      l.vendor.vertical_type
    ) AS vertical_type,

    CASE
      WHEN l.food_is_ready_at IS NOT NULL
      THEN TIMESTAMP_DIFF(
        l.food_is_ready_at,
        TIMESTAMP(l.created_at_local),
        SECOND
      )
    END AS fir_seconds,

    DATE(l.created_date_local) AS order_date,

    EXTRACT(
      DAYOFWEEK
      FROM DATE(l.created_date_local)
    ) AS day_of_week_num,

    FORMAT_DATE(
      '%A',
      DATE(l.created_date_local)
    ) AS day_of_week_name,

    CASE
      WHEN EXTRACT(HOUR FROM l.created_at_local)
        BETWEEN 12 AND 14
        THEN 'lunch'

      WHEN EXTRACT(HOUR FROM l.created_at_local)
        BETWEEN 19 AND 21
        THEN 'dinner'

      ELSE 'valle'
    END AS time_block,

    SAFE_DIVIDE(
      l.estimated_prep_time,
      60
    ) AS ept_min,

    SAFE_DIVIDE(
      l.timings.avoidable_wait_time,
      60
    ) AS awt_min,

    TIMESTAMP_DIFF(
      d.rider_picked_up_at_local,
      l.created_at_local,
      MINUTE
    ) AS created_to_pickup,

    o.is_slow_order AS slow,
    o.non_seamless_order AS non_seamless,

    h.order_code AS hd_order_code,
    h.minutes_added AS hd_minutes_added,
    h.duration AS hd_duration,
    h.enabled_at AS hd_enabled_at,
    h.disabled_at AS hd_disabled_at

  FROM `peya-bi-tools-pro.il_logistics.fact_logistic_orders` l

  LEFT JOIN UNNEST(l.deliveries) d

  INNER JOIN target_stores p
    ON l.vendor.vendor_code = p.vendor_code

  LEFT JOIN `peya-datamarts-pro.dm_fulfillment.non_seamless_delivery_order_level` o
    ON o.platform_order_code = l.peya_order_id

  LEFT JOIN hdm h
    ON h.order_code = SAFE_CAST(
      l.peya_order_id AS INT64
    )

  WHERE l.country_code = 'cl'
    -- AND l.vendor.vertical_type = 'restaurants'
    AND l.is_preorder = FALSE

    AND l.created_date_local
      BETWEEN DATE '{start_date}'
          AND DATE '{end_date}'
),

zero_food_ready_franchises AS (
  SELECT
    franchise_group_id,
    franchise_id,
    franchise_name,

    COUNT(
      DISTINCT peya_order_id
    ) AS franchise_orders,

    COUNT(
      DISTINCT vendor_code
    ) AS franchise_stores,

    COUNT(
      DISTINCT IF(
        food_is_ready_at IS NOT NULL,
        peya_order_id,
        NULL
      )
    ) AS franchise_orders_with_fir,

    SAFE_DIVIDE(
      COUNT(
        DISTINCT IF(
          food_is_ready_at IS NOT NULL,
          peya_order_id,
          NULL
        )
      ),
      COUNT(DISTINCT peya_order_id)
    ) AS franchise_fir

  FROM base

  GROUP BY
    franchise_group_id,
    franchise_id,
    franchise_name
),

valid_stores AS (
  SELECT
    b.franchise_group_id,
    b.franchise_id,
    b.franchise_name,
    b.city_name,
    b.vertical_type,

    z.franchise_orders,
    z.franchise_stores,
    z.franchise_orders_with_fir,
    z.franchise_fir,

    b.vendor_code,
    b.store_name,

    COUNT(
      DISTINCT b.peya_order_id
    ) AS store_orders_total,

    COUNT(
      DISTINCT IF(
        b.food_is_ready_at IS NOT NULL,
        b.peya_order_id,
        NULL
      )
    ) AS store_orders_with_fir,

    SAFE_DIVIDE(
      COUNT(
        DISTINCT IF(
          b.food_is_ready_at IS NOT NULL,
          b.peya_order_id,
          NULL
        )
      ),
      COUNT(DISTINCT b.peya_order_id)
    ) AS store_fir

  FROM base b

  INNER JOIN zero_food_ready_franchises z
    ON b.franchise_group_id = z.franchise_group_id

  GROUP BY
    b.franchise_group_id,
    b.franchise_id,
    b.franchise_name,
    b.city_name,
    b.vertical_type,

    z.franchise_orders,
    z.franchise_stores,
    z.franchise_orders_with_fir,
    z.franchise_fir,

    b.vendor_code,
    b.store_name
),

store_day_block_metrics AS (
  SELECT
    b.franchise_group_id,
    b.franchise_id,
    b.vendor_code,
    b.city_name,
    b.vertical_type,

    b.day_of_week_num,
    b.day_of_week_name,
    b.time_block,

    SUM(
      b.fir_seconds
    ) AS sum_fir_seconds,

    ROUND(
      SAFE_DIVIDE(
        SUM(b.fir_seconds),
        60
      ),
      2
    ) AS sum_fir_min,

    ROUND(
      SAFE_DIVIDE(
        AVG(b.fir_seconds),
        60
      ),
      2
    ) AS avg_fir_min,

    COUNT(
      DISTINCT b.peya_order_id
    ) AS total_orders,

    COUNT(
      DISTINCT IF(
        b.slow = 1,
        b.peya_order_id,
        NULL
      )
    ) AS slow_orders,

    COUNT(
      DISTINCT IF(
        b.non_seamless = TRUE,
        b.peya_order_id,
        NULL
      )
    ) AS non_seamless_orders,

    COUNT(
      DISTINCT IF(
        b.food_is_ready_at IS NOT NULL,
        b.peya_order_id,
        NULL
      )
    ) AS orders_with_fir,

    SAFE_DIVIDE(
      COUNT(
        DISTINCT IF(
          b.food_is_ready_at IS NOT NULL,
          b.peya_order_id,
          NULL
        )
      ),
      COUNT(DISTINCT b.peya_order_id)
    ) AS fir,

    ROUND(
      AVG(b.ept_min),
      2
    ) AS ept,

    ROUND(
      AVG(b.awt_min),
      2
    ) AS awt,

    ROUND(
      AVG(b.ept_min) + AVG(b.awt_min),
      2
    ) AS tt,

    ROUND(
      AVG(b.created_to_pickup),
      2
    ) AS ctp

  FROM base b

  INNER JOIN valid_stores s
    ON b.franchise_group_id = s.franchise_group_id
   AND b.vendor_code = s.vendor_code
   AND b.city_name = s.city_name
   AND b.vertical_type = s.vertical_type

  GROUP BY
    b.franchise_group_id,
    b.franchise_id,
    b.vendor_code,
    b.city_name,
    b.vertical_type,
    b.day_of_week_num,
    b.day_of_week_name,
    b.time_block
),

hdm_order_metrics AS (
  SELECT
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,

    day_of_week_num,
    day_of_week_name,
    time_block,

    COUNT(
      DISTINCT peya_order_id
    ) AS hd_total_orders,

    ROUND(
      SUM(hd_minutes_added),
      2
    ) AS hd_total_minutes_added

  FROM (
    SELECT
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,

      day_of_week_num,
      day_of_week_name,
      time_block,

      peya_order_id,

      MAX(
        COALESCE(
          hd_minutes_added,
          0
        )
      ) AS hd_minutes_added

    FROM base

    WHERE hd_order_code IS NOT NULL

    GROUP BY
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,
      day_of_week_num,
      day_of_week_name,
      time_block,
      peya_order_id
  )

  GROUP BY
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,
    day_of_week_num,
    day_of_week_name,
    time_block
),

hdm_trigger_metrics AS (
  SELECT
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,

    day_of_week_num,
    day_of_week_name,
    time_block,

    COUNT(*) AS started_triggers,

    ROUND(
      AVG(duration),
      2
    ) AS avg_trigger_duration_min

  FROM (
    SELECT
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,

      EXTRACT(
        DAYOFWEEK
        FROM DATE(hd_enabled_at)
      ) AS day_of_week_num,

      FORMAT_DATE(
        '%A',
        DATE(hd_enabled_at)
      ) AS day_of_week_name,

      CASE
        WHEN EXTRACT(HOUR FROM hd_enabled_at)
          BETWEEN 12 AND 14
          THEN 'lunch'

        WHEN EXTRACT(HOUR FROM hd_enabled_at)
          BETWEEN 19 AND 21
          THEN 'dinner'

        ELSE 'valle'
      END AS time_block,

      hd_enabled_at,

      MAX(
        hd_duration
      ) AS duration

    FROM base

    WHERE hd_enabled_at IS NOT NULL

    GROUP BY
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,
      day_of_week_num,
      day_of_week_name,
      time_block,
      hd_enabled_at
  )

  GROUP BY
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,
    day_of_week_num,
    day_of_week_name,
    time_block
)

SELECT
  t.franchise_id,
  t.franchise_name,
  s.city_name,
  COALESCE(
    s.vertical_type,
    t.vertical_type
  ) AS vertical_type,

  s.franchise_orders,
  s.franchise_stores,
  s.franchise_orders_with_fir,

  ROUND(
    s.franchise_fir,
    4
  ) AS franchise_fir,

  t.vendor_code,

  COALESCE(
    s.store_name,
    t.store_name
  ) AS store_name,

  s.store_orders_total,
  s.store_orders_with_fir,

  ROUND(
    s.store_fir,
    4
  ) AS store_fir,

  m.day_of_week_num,
  m.day_of_week_name,
  m.time_block,

  m.total_orders,
  m.slow_orders,
  m.non_seamless_orders,

  COALESCE(
    hdo.hd_total_orders,
    0
  ) AS hd_total_orders,

  COALESCE(
    hdo.hd_total_minutes_added,
    0
  ) AS hd_total_minutes_added,

  COALESCE(
    hdt.started_triggers,
    0
  ) AS started_triggers,

  hdt.avg_trigger_duration_min,

  m.orders_with_fir,

  ROUND(
    m.fir,
    4
  ) AS fir_ratio,

  m.ept,
  m.awt,
  m.tt,
  m.ctp,
  m.avg_fir_min AS fir

FROM target_stores t

LEFT JOIN valid_stores s
  ON t.franchise_group_id = s.franchise_group_id
 AND t.vendor_code = s.vendor_code

LEFT JOIN store_day_block_metrics m
  ON t.franchise_group_id = m.franchise_group_id
 AND s.vendor_code = m.vendor_code
 AND s.city_name = m.city_name
 AND s.vertical_type = m.vertical_type

LEFT JOIN hdm_order_metrics hdo
  ON s.franchise_group_id = hdo.franchise_group_id
 AND s.vendor_code = hdo.vendor_code
 AND s.city_name = hdo.city_name
 AND s.vertical_type = hdo.vertical_type
 AND m.day_of_week_num = hdo.day_of_week_num
 AND m.time_block = hdo.time_block

LEFT JOIN hdm_trigger_metrics hdt
  ON s.franchise_group_id = hdt.franchise_group_id
 AND s.vendor_code = hdt.vendor_code
 AND s.city_name = hdt.city_name
 AND s.vertical_type = hdt.vertical_type
 AND m.day_of_week_num = hdt.day_of_week_num
 AND m.time_block = hdt.time_block

ORDER BY
  t.franchise_name,
  COALESCE(
    s.store_name,
    t.store_name
  ),
  m.day_of_week_num,

  CASE m.time_block
    WHEN 'lunch' THEN 1
    WHEN 'dinner' THEN 2
    WHEN 'valle' THEN 3
  END
"""


# print(
#     f"Reglas individuales: {len(vendor_codes)} | "
#     f"Reglas por franquicia: {len(franchise_ids)} | "
#     f"Fechas: {start_date} a {end_date}"
# )

print(query)


WITH vendor_verticals AS (
  SELECT
    vendor_code,
    ARRAY_AGG(
      vertical_type IGNORE NULLS
      ORDER BY vertical_type
      LIMIT 1
    )[SAFE_OFFSET(0)] AS vertical_type

  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendors`

  WHERE entity_id = 'PY_CL'
    AND vendor_code IS NOT NULL

  GROUP BY vendor_code
),

hdm AS (
  SELECT
    SAFE_CAST(order_code AS INT64) AS order_code,

    high_demand_mode.is_hd_order AS hd_order,
    SAFE_CAST(
      high_demand_mode.minutes_added AS FLOAT64
    ) AS minutes_added,
    SAFE_CAST(
      high_demand_mode.duration AS FLOAT64
    ) AS duration,

    high_demand_mode.enabled_at,
    high_demand_mode.disabled_at

  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders`

  WHERE created_date BETWEEN DATE '2026-09-03'
                         AND DATE '2026-09-09'
    AND country_code = 'cl'
    AND high_demand_mode.is_hd_order IS TRUE
    AND requested_pickup_at IS NULL
),

target_st

In [7]:
# input_confirmation = input("¿Ya tomaste los resultados de la query y actualizaste los EPT actuales en la hoja de cálculo?  \n https://docs.google.com/spreadsheets/d/1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE \n Escribe 'sí' para continuar: ")

# if input_confirmation.lower() != 'si':
#     raise ValueError("Por favor, actualiza los EPTs en la hoja de cálculo y vuelve a ejecutar esta celda.")

# print("Confirmación recibida. Continuando con el procesamiento.")

### ept_actuales

In [8]:
# # EPT actuales
# file_name = 'Wave week36.csv'

# from google.colab import drive
# drive.mount('/content/drive')
# file_path_main = '/content/drive/MyDrive/lower ept y awt/ept reductions input data/'
# file_path = file_path_main + file_name
# ept_actuales_raw = pd.read_csv(file_path)
# ept_actuales_raw.head(3)

In [9]:
from google.cloud import bigquery

project_id = "peya-chile"

# Ejecuta la consulta en BigQuery y descarga los datos
bq_client = bigquery.Client(project=project_id, credentials=credentials)
ept_actuales_raw = bq_client.query(query).to_dataframe()

# Muestra las primeras 5 filas del resultado
ept_actuales_raw.head()

c:\Users\pablo.villanueva\.gemini\antigravity\scratch\bigquery_pipeline\github primer repo\lower-ept-y-awt\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,franchise_id,franchise_name,city_name,vertical_type,franchise_orders,franchise_stores,franchise_orders_with_fir,franchise_fir,vendor_code,store_name,store_orders_total,store_orders_with_fir,store_fir,day_of_week_num,day_of_week_name,time_block,total_orders,slow_orders,non_seamless_orders,hd_total_orders,hd_total_minutes_added,started_triggers,avg_trigger_duration_min,orders_with_fir,fir_ratio,ept,awt,tt,ctp,fir
0,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,lunch,14,0,4,0,0.0,0,NaN,14,1.0,8.29,1.00,9.29,11.43,187.76
1,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,7.00,0.00,7.00,7.00,184.25
2,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,valle,16,0,0,0,0.0,0,NaN,16,1.0,8.50,6.12,14.62,16.56,192.74
3,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,3,Tuesday,lunch,15,1,4,0,0.0,0,NaN,15,1.0,9.07,3.52,12.58,19.40,190.89
4,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,3,Tuesday,dinner,5,0,0,0,0.0,0,NaN,5,1.0,9.20,0.12,9.32,8.60,187.35


In [10]:
# # 1. Prepara el DataFrame: gspread no acepta valores NaN ni formatos complejos de fecha
# df_to_export = ept_actuales_raw.fillna("")
# df_to_export = df_to_export.astype(str) # Convertimos todo a texto para evitar errores de formato

# # 2. Abre el documento de Google Sheets (usa el mismo sheet_id que ya tienes u otro)
# # gc ya está autorizado en las celdas anteriores
# sheet_id = "TU_ID_DEL_DOCUMENTO_AQUI"
# spreadsheet = gc.open_by_key(sheet_id)

# # 3. Crea una nueva pestaña para los resultados de la query
# nombre_pestaña = "Resultados Query"
# try:
#     worksheet = spreadsheet.worksheet(nombre_pestaña)
#     worksheet.clear() # Limpia la pestaña si ya existe
# except gspread.WorksheetNotFound:
#     worksheet = spreadsheet.add_worksheet(title=nombre_pestaña, rows=100, cols=20)

# # 4. Convierte el DataFrame a una lista de listas (encabezados + valores)
# data = [df_to_export.columns.values.tolist()] + df_to_export.values.tolist()

# # 5. Escribe los datos en la hoja
# worksheet.update(range_name="A1", values=data, value_input_option="RAW")
# print(f"¡Exportación exitosa a la pestaña '{nombre_pestaña}'!")

In [11]:
# import gspread
# from google.colab import auth
# import google.auth

# auth.authenticate_user()

# credentials, _ = google.auth.default(scopes=[
#     "https://www.googleapis.com/auth/spreadsheets",
#     "https://www.googleapis.com/auth/drive"
# ])

# gc = gspread.authorize(credentials)
# sheet_id = "1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE"
# # manual ept modifications
# # https://docs.google.com/spreadsheets/d/1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE/edit?gid=0#gid=0
# sheet_name = "EPT actuales"

# worksheet = gc.open_by_key(sheet_id).worksheet(sheet_name)
# data = worksheet.get_all_values()
# ept_actuales_raw = pd.DataFrame(data[1:], columns=data[0])

# ept_actuales_raw.head()

In [12]:
ept_actuales = ept_actuales_raw.copy()
ept_actuales.head(3) # agregar a query los filtros de la query anterior, incluir city_name, slow_orders, non_seamless_orders, ctp

,franchise_id,franchise_name,city_name,vertical_type,franchise_orders,franchise_stores,franchise_orders_with_fir,franchise_fir,vendor_code,store_name,store_orders_total,store_orders_with_fir,store_fir,day_of_week_num,day_of_week_name,time_block,total_orders,slow_orders,non_seamless_orders,hd_total_orders,hd_total_minutes_added,started_triggers,avg_trigger_duration_min,orders_with_fir,fir_ratio,ept,awt,tt,ctp,fir
0,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,lunch,14,0,4,0,0.0,0,NaN,14,1.0,8.29,1.00,9.29,11.43,187.76
1,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,7.00,0.00,7.00,7.00,184.25
2,NaN,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,valle,16,0,0,0,0.0,0,NaN,16,1.0,8.50,6.12,14.62,16.56,192.74


#### vendors validation

In [13]:
# Validar que las reglas por franquicia sí se hayan podido expandir.
# Los vendor_code individuales pueden no tener historia y se completan más adelante.
ept_actuales["vendor_code"] = normalize_id(ept_actuales["vendor_code"])
ept_actuales["franchise_id"] = normalize_id(ept_actuales["franchise_id"])

franchises_expected = set(franchise_ids)
franchises_found = set(ept_actuales["franchise_id"].dropna())
missing_franchises = sorted(franchises_expected - franchises_found)

if missing_franchises:
    raise ValueError(
        "No se encontraron vendors online para estos franchise_id: "
        + ", ".join(missing_franchises)
    )

print(
    f"Franquicias expandidas: {len(franchises_expected)} | "
    f"Vendors obtenidos: {ept_actuales['vendor_code'].nunique()}"
)

Franquicias expandidas: 0 | Vendors obtenidos: 11


#### agrupado por vendor

In [14]:
import numpy as np

def wavg(x, value_col, weight_col='total_orders'):
    """
    Promedio ponderado usando únicamente filas con valor y peso válidos.
    """
    values = pd.to_numeric(x[value_col], errors='coerce')
    weights = pd.to_numeric(x[weight_col], errors='coerce')

    mask = values.notna() & weights.notna() & (weights > 0)

    if not mask.any() or weights.loc[mask].sum() == 0:
        return np.nan

    return np.average(
        values.loc[mask],
        weights=weights.loc[mask]
    )


def weighted_total(x, value_col, weight_col='total_orders'):
    """
    Convierte un promedio por fila en volumen total:
        sum(value * weight)
    """
    values = pd.to_numeric(x[value_col], errors='coerce')
    weights = pd.to_numeric(x[weight_col], errors='coerce')

    mask = values.notna() & weights.notna() & (weights > 0)

    if not mask.any():
        return 0.0

    return (values.loc[mask] * weights.loc[mask]).sum()

In [15]:
group_cols = [
    'vendor_code'
]

df_grouped = (
    ept_actuales
    .groupby(group_cols, as_index=False)
    .apply(lambda x: pd.Series({

        'franchise_id': x['franchise_id'].max(),
        'franchise_name': x['franchise_name'].max(),
        'city_name': x['city_name'].max(),
        'vertical_type': (
            x['vertical_type'].dropna().iloc[0]
            if x['vertical_type'].notna().any() else np.nan
        ),
        'store_name': x['store_name'].max(),
        'franchise_orders': pd.to_numeric(x['franchise_orders'], errors='coerce').iloc[0],
        'franchise_stores': pd.to_numeric(x['franchise_stores'], errors='coerce').iloc[0],
        'store_orders_total': pd.to_numeric(x['store_orders_total'], errors='coerce').iloc[0],

        # Volúmenes - explicit conversion to numeric before summing
        'total_orders': pd.to_numeric(x['total_orders'], errors='coerce').sum(),
        'slow_orders': pd.to_numeric(x['slow_orders'], errors='coerce').sum(),
        'non_seamless_orders': pd.to_numeric(x['non_seamless_orders'], errors='coerce').sum(),
        'orders_with_fir': pd.to_numeric(x['orders_with_fir'], errors='coerce').sum(),

        # HDM - explicit conversion to numeric before summing
        # En la data raw la columna se llama hd_total_orders.
        'hd_orders': pd.to_numeric(x['hd_total_orders'], errors='coerce').sum(),
        'hd_total_minutes_added': pd.to_numeric(x['hd_total_minutes_added'], errors='coerce').sum(),

        # Total de minutos EPT antes de quitar HDM.
        'ept_total_minutes': weighted_total(
            x,
            value_col='ept',
            weight_col='total_orders'
        ),

        # Promedios
        'ept': wavg(x, 'ept', 'total_orders'),
        'awt': wavg(x, 'awt', 'total_orders'),
        'tt': wavg(x, 'tt', 'total_orders'),

        # FIR promedio entre las órdenes que efectivamente tienen FIR.
        # No se pondera por total_orders, sino por orders_with_fir.
        'fir': wavg(x, 'fir', 'orders_with_fir'),

        'ctp': wavg(x, 'ctp', 'total_orders')
    }), include_groups=False)
    .reset_index(drop=True)
)

df_grouped['slow_ratio'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['slow_orders'] / df_grouped['total_orders'],
    np.nan
)

df_grouped['non_seamless_ratio'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['non_seamless_orders'] / df_grouped['total_orders'],
    np.nan
)

# FIR ratio del vendor: share de órdenes que tienen FIR.
df_grouped['fir_ratio'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['orders_with_fir'] / df_grouped['total_orders'],
    np.nan
)

# Nombre explícito para el FIR promedio en minutos.
df_grouped['fir_avg'] = df_grouped['fir']

# EPT promedio quitando los minutos agregados por HDM:
# (sum(ept * total_orders) - sum(hd_total_minutes_added)) / sum(total_orders)
df_grouped['avg_ept_sin_hdm'] = np.where(
    df_grouped['total_orders'] > 0,
    (
        df_grouped['ept_total_minutes']
        - df_grouped['hd_total_minutes_added']
    ) / df_grouped['total_orders'],
    np.nan
)

# Control: diferencia promedio explicada por HDM.
df_grouped['avg_hdm_minutes_per_order'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['hd_total_minutes_added'] / df_grouped['total_orders'],
    np.nan
)

df_grouped = df_grouped.sort_values(
    ['franchise_name', 'store_name', 'city_name']
).reset_index(drop=True)


df_grouped['ept_raw'] = df_grouped['ept']
df_grouped['ept'] = df_grouped['avg_ept_sin_hdm']
df_grouped_raw = df_grouped.copy()

In [16]:
df_grouped_raw

,vendor_code,franchise_id,franchise_name,city_name,vertical_type,store_name,franchise_orders,franchise_stores,store_orders_total,total_orders,slow_orders,non_seamless_orders,orders_with_fir,hd_orders,hd_total_minutes_added,ept_total_minutes,ept,awt,tt,fir,ctp,slow_ratio,non_seamless_ratio,fir_ratio,fir_avg,avg_ept_sin_hdm,avg_hdm_minutes_per_order,ept_raw
0,519467,0016900002xQv80AAC,Club Del Poke - LATAM,Copiapo,restaurants,Club Del Poke - Copiapó.,93,1,93,93,11,30,81,16,160.0,1242.04,11.634839,11.683763,25.038387,232.818765,28.836129,0.118280,0.322581,0.870968,232.818765,11.634839,1.720430,13.355269
1,535773,0011r00002VoHw6AAF,Papa John's,Osorno,restaurants,Papa John's Rahue,2612,5,329,329,47,72,0,60,600.0,9558.68,27.230030,2.744847,31.851043,NaN,35.775137,0.142857,0.218845,0.000000,NaN,27.230030,1.823708,29.053739
2,341835,0011r00002VoHw6AAF,Papa John's,Copiapo,restaurants,Papa Johns Pizza - Copiapo,2612,5,550,550,77,105,0,0,0.0,15685.36,28.518836,2.949691,31.469545,NaN,33.758982,0.140000,0.190909,0.000000,NaN,28.518836,0.000000,28.518836
3,133603,0011r00002VoHw6AAF,Papa John's,Puerto montt,restaurants,Papa Johns Pizza - Puerto Montt,2612,5,485,485,63,89,0,80,800.0,13923.29,27.058330,2.981299,31.689794,NaN,35.783546,0.129897,0.183505,0.000000,NaN,27.058330,1.649485,28.707814
4,133509,0011r00002VoHw6AAF,Papa John's,Punta arenas,restaurants,Papa Johns Pizza - Punta Arenas,2612,5,643,643,205,226,0,200,2000.0,22713.33,32.213577,3.602768,38.925023,NaN,41.668834,0.318818,0.351477,0.000000,NaN,32.213577,3.110420,35.323997
5,291016,0011r00002VoHw6AAF,Papa John's,Concepcion,restaurants,Papa John´s - Coronel,2612,5,605,605,125,159,0,166,1556.0,20736.77,31.703752,2.982876,37.257967,NaN,39.479736,0.206612,0.262810,0.000000,NaN,31.703752,2.571901,34.275653
6,301949,NaN,NaN,Rancagua,restaurants,Gohan Club,156,1,156,156,14,36,155,0,0.0,1424.08,9.128718,6.572885,15.700128,225.157613,19.061026,0.089744,0.230769,0.993590,225.157613,9.128718,0.000000,9.128718
7,286542,NaN,NaN,Antofagasta,restaurants,Mi Delizie Sur Sushi,701,1,701,701,276,291,104,28,240.0,22396.77,31.607375,1.773367,33.721840,242.442404,46.000699,0.393723,0.415121,0.148359,242.442404,31.607375,0.342368,31.949743
8,418988,NaN,NaN,Concepcion,restaurants,Nori & Nigiri,85,1,85,85,24,32,10,3,24.0,3119.96,36.423059,4.780250,41.593875,241.618000,47.599647,0.282353,0.376471,0.117647,241.618000,36.423059,0.282353,36.705412
9,87771,NaN,NaN,Iquique,restaurants,Restaurante Angaroa,15,1,15,15,10,10,8,0,0.0,419.98,27.998667,13.245000,36.496250,248.236250,52.665333,0.666667,0.666667,0.533333,248.236250,27.998667,0.000000,27.998667


## Funciones

### weighted_percentile()

In [17]:
def weighted_percentile(group, value_col, weight_col, percentile_val):
    values = group[value_col].to_numpy()
    weights = group[weight_col].to_numpy()

    # Filter out NaNs from values and weights, and weights that are zero or negative
    valid_mask = ~np.isnan(values) & ~np.isnan(weights) & (weights > 0)
    values = values[valid_mask]
    weights = weights[valid_mask]

    if len(values) == 0:
        return np.nan

    # Sort values and corresponding weights
    idx = np.argsort(values)
    sorted_values = values[idx]
    sorted_weights = weights[idx]

    # Calculate cumulative sum of weights
    cumulative_weights = np.cumsum(sorted_weights)
    total_weight = cumulative_weights[-1]

    if total_weight == 0: # This case should be rare after filtering, but for safety
        return np.nan

    # Find the index where the cumulative weight exceeds the threshold
    threshold_weight = total_weight * (percentile_val / 100.0)

    # Use searchsorted to find the index of the first value whose cumulative weight
    # is greater than or equal to the threshold_weight.
    interp_idx = np.searchsorted(cumulative_weights, threshold_weight, side='left')

    # Handle edge cases:
    if percentile_val == 0:
        return sorted_values[0]
    if percentile_val == 100:
        return sorted_values[-1]
    if interp_idx >= len(sorted_values): # Should only happen if percentile_val is 100
        return sorted_values[-1]

    return sorted_values[interp_idx]

median_cap()

In [18]:
def median_cap(
    df,
    value_col="ept_new",
    weight_col="total_orders",
    group_col="vendor_code",
    percentile=50,
    multiplier=1.2
):
    """
    Capea value_col a multiplier * weighted_percentile(group).

    Retorna una copia del dataframe con:
        - vendor_percentile
        - upper_limit
        - is_capped
        - value_col modificado
    """

    new_df = df.copy()

    percentile_df = (
        new_df
        .groupby(group_col)
        .apply(
            lambda x: weighted_percentile(
                x,
                value_col,
                weight_col,
                percentile
            ),
            include_groups=False
        )
        .reset_index(name="vendor_percentile")
    )

    new_df = new_df.merge(
        percentile_df,
        on=group_col,
        how="left"
    )

    new_df["upper_limit"] = (
        new_df["vendor_percentile"] * multiplier
    )

    new_df["is_capped"] = (
        new_df[value_col] > new_df["upper_limit"]
    )

    new_df[value_col] = np.where(
        new_df["is_capped"],
        new_df["upper_limit"],
        new_df[value_col]
    )

    print(f"Number of rows capped: {new_df['is_capped'].sum():,}")

    return new_df

### aplicar_factor_ept()

In [19]:
import numpy as np
import pandas as pd

def aplicar_factor_ept(ept_actuales, df_reduction):
    df_out = ept_actuales.copy()

    store_col = "vendor_code"
    ept_col = "ept"
    reduction_col = "ept_reduction_pct"

    # Asegurar mismo tipo para merge
    df_out[store_col] = df_out[store_col].astype(str)

    factors = (
        df_reduction[[store_col, reduction_col]]
        .copy()
    )

    factors[store_col] = factors[store_col].astype(str)

    # Si viene como 13 en vez de 0.13, lo transforma a 0.13
    factors[reduction_col] = np.where(
        factors[reduction_col] > 1,
        factors[reduction_col] / 100,
        factors[reduction_col]
    )

    # Una fila por vendor_code
    factors = (
        factors
        .drop_duplicates(subset=[store_col])
        .rename(columns={reduction_col: "ept_reduction_pct_applied"})
    )

    # Merge left para NO perder filas de ept_actuales
    df_out = df_out.merge(
        factors,
        on=store_col,
        how="left"
    )

    # Vendors sin factor quedan iguales. No se hace clip porque también
    # se permiten aumentos de EPT (porcentaje de reducción negativo).
    df_out["ept_reduction_pct_applied"] = (
        pd.to_numeric(
            df_out["ept_reduction_pct_applied"],
            errors="coerce"
        ).fillna(0)
    )

    # Aplicar reducción
    df_out["ept_new"] = (
        df_out[ept_col] * (1 - df_out["ept_reduction_pct_applied"])
    )

    df_out["ept_delta_min"] = df_out["ept_new"] - df_out[ept_col]

    print(f"Avg EPT actual: {df_out[ept_col].mean():.2f}")
    print(f"Avg EPT nuevo:  {df_out['ept_new'].mean():.2f}")
    print(f"Delta avg EPT:  {df_out['ept_new'].mean() - df_out[ept_col].mean():.2f}")

    return df_out

### exportar_template_ops()

In [20]:
import os
import pandas as pd
from datetime import datetime

def exportar_template_ops(ept_actuales_sim, max_num=999):
    if not 1 <= max_num < 1000:
        raise ValueError('max_num debe estar entre 1 y 999 filas.')

    df = ept_actuales_sim.copy()

    if os.path.exists('/content/drive/MyDrive'):
        results_folder = '/content/drive/MyDrive/lower ept y awt/results'
    else:
        results_folder = os.path.join(os.getcwd(), 'results')
    os.makedirs(results_folder, exist_ok=True)

    df['day_of_week_name_clean'] = df['day_of_week_name'].astype(str).str.upper().str.strip()

    day_map = {
        'MONDAY': 'MONDAY-MONDAY',
        'TUESDAY': 'TUESDAY-TUESDAY',
        'WEDNESDAY': 'WEDNESDAY-WEDNESDAY',
        'THURSDAY': 'THURSDAY-THURSDAY',
        'FRIDAY': 'FRIDAY-FRIDAY',
        'SATURDAY': 'SATURDAY-SATURDAY',
        'SUNDAY': 'SUNDAY-SUNDAY',
        'LUNES': 'MONDAY-MONDAY',
        'MARTES': 'TUESDAY-TUESDAY',
        'MIERCOLES': 'WEDNESDAY-WEDNESDAY',
        'MIÉRCOLES': 'WEDNESDAY-WEDNESDAY',
        'JUEVES': 'THURSDAY-THURSDAY',
        'VIERNES': 'FRIDAY-FRIDAY',
        'SABADO': 'SATURDAY-SATURDAY',
        'SÁBADO': 'SATURDAY-SATURDAY',
        'DOMINGO': 'SUNDAY-SUNDAY'
    }

    df['DAY-RANGE'] = df['day_of_week_name_clean'].map(day_map)

    df['time_block_clean'] = df['time_block'].astype(str).str.upper().str.strip()

    hour_map = {
        'LUNCH': ['12-14'],
        'DINNER': ['19-21'],
        'VALLE': ['0-11', '15-18', '22-23'],
        'OFF_PEAK': ['0-11', '15-18', '22-23'],
        'RESTO': ['0-11', '15-18', '22-23']
    }

    df['HOUR-RANGE'] = df['time_block_clean'].map(hour_map)
    df = df.explode('HOUR-RANGE')

    df_template = pd.DataFrame({
        'CODE': df['vendor_code'].astype(str),
        'DAY-RANGE': df['DAY-RANGE'].astype(str),
        # Fórmula de texto para que Google Sheets NO lo transforme a fecha
        'HOUR-RANGE': df['HOUR-RANGE'].astype(str),
        'PREPARATION-BUFFER': 2,
        'PREPARATION-TIME': df['ept_new'].astype(int),
        'STRATEGY': 'OPS_TEMPORARY'
    })

    # Ordenar para que las filas de cada CODE queden juntas
    df_template = df_template.sort_values(
        ['CODE', 'DAY-RANGE', 'HOUR-RANGE']
    ).reset_index(drop=True)

    # Validar que ningún CODE supere max_num filas por sí solo
    rows_by_code = df_template.groupby('CODE').size()

    codes_over_limit = rows_by_code[rows_by_code > max_num]

    if len(codes_over_limit) > 0:
        raise ValueError(
            f"Hay CODEs con más de max_num filas, imposible mantenerlos completos en un solo archivo: "
            f"{codes_over_limit.to_dict()}"
        )

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    export_folder = os.path.join(
        results_folder, f'template_ops_tiempos_{timestamp}'
    )
    os.makedirs(export_folder, exist_ok=False)

    files_created = []
    current_chunk = []
    current_rows = 0
    part = 1

    for code, df_code in df_template.groupby('CODE', sort=False):
        code_rows = len(df_code)

        if current_rows + code_rows > max_num:
            chunk_df = pd.concat(current_chunk, ignore_index=True)

            filename = f'part_{part:02d}.csv'
            output_path = os.path.join(export_folder, filename)

            chunk_df.to_csv(output_path, index=False)

            files_created.append(output_path)

            part += 1
            current_chunk = []
            current_rows = 0

        current_chunk.append(df_code)
        current_rows += code_rows

    # Guardar último chunk
    if current_chunk:
        chunk_df = pd.concat(current_chunk, ignore_index=True)

        filename = f'part_{part:02d}.csv'
        output_path = os.path.join(export_folder, filename)

        chunk_df.to_csv(output_path, index=False)

        files_created.append(output_path)

    print(f"Filas totales exportadas: {len(df_template):,}")
    print(f"Archivos creados: {len(files_created):,}")
    print(f"Carpeta creada: {export_folder}")

    for path in files_created:
        print(path)

    df_template.attrs['export_folder'] = export_folder
    df_template.attrs['export_files'] = files_created
    return df_template

### rellenar_vendor_dow_block()

In [21]:
import numpy as np
import pandas as pd

def rellenar_vendor_dow_block(
    df,
    target_time_blocks=['lunch', 'dinner', 'valle']
):
    df = df.copy()

    vendor_col = 'vendor_code'
    dow_num_col = 'day_of_week_num'
    dow_name_col = 'day_of_week_name'
    block_col = 'time_block'
    weight_col = 'total_orders'

    # Explicitly convert day_of_week_num to numeric
    df[dow_num_col] = pd.to_numeric(df[dow_num_col], errors='coerce')

    days_df = pd.DataFrame({
        dow_num_col: [1, 2, 3, 4, 5, 6, 7],
        dow_name_col: ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    })

    meta_cols = [
        'franchise_id', 'franchise_name', 'city_name', 'vertical_type',
        'franchise_orders', 'franchise_stores',
        'store_name', 'store_orders_total'
    ]

    count_cols = [
        'total_orders', 'slow_orders', 'non_seamless_orders'
    ]

    metric_cols = [
        'ept', 'awt', 'tt', 'ctp'
    ]

    meta_cols = [c for c in meta_cols if c in df.columns]
    count_cols = [c for c in count_cols if c in df.columns]
    metric_cols = [c for c in metric_cols if c in df.columns]

    for col in count_cols + metric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    def first_notna(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else np.nan

    def wavg(g, col):
        mask = g[col].notna()

        if weight_col in g.columns:
            mask = mask & g[weight_col].notna() & (g[weight_col] > 0)

            if mask.sum() > 0:
                return np.average(g.loc[mask, col], weights=g.loc[mask, weight_col])

        return g[col].mean()

    def summarize(keys, suffix=None):
        rows = []

        for key_values, g in df.groupby(keys, dropna=False):
            if not isinstance(key_values, tuple):
                key_values = (key_values,)

            row = dict(zip(keys, key_values))

            for col in metric_cols:
                out_col = f'{col}_{suffix}' if suffix else col
                row[out_col] = wavg(g, col)

            rows.append(row)

        return pd.DataFrame(rows)

    # 1) Metadata única por vendor
    vendor_meta = (
        df[[vendor_col] + meta_cols]
        .groupby(vendor_col, as_index=False)
        .agg({c: first_notna for c in meta_cols})
    )

    # 2) Grilla completa vendor x dow x block
    all_blocks_df = pd.DataFrame({block_col: target_time_blocks})

    grid = (
        df[[vendor_col]]
        .drop_duplicates()
        .merge(days_df, how='cross')
        .merge(all_blocks_df, how='cross')
    )

    # 3) Data real agregada por vendor + dow + block
    real_rows = []
    group_keys = [vendor_col, dow_num_col, dow_name_col, block_col]

    # Define all possible columns for real_df including metrics and auxiliary flag
    real_df_cols = group_keys + count_cols + metric_cols + ['_is_real_row']

    # Only iterate if df is not empty to avoid issues with groupby on empty DataFrame
    if not df.empty:
        for key_values, g in df.groupby(group_keys, dropna=False):
            row = dict(zip(group_keys, key_values))

            for col in count_cols:
                row[col] = g[col].sum(min_count=1)

            for col in metric_cols:
                row[col] = wavg(g, col)

            row['_is_real_row'] = 1
            real_rows.append(row)

    # Ensure real_df has defined columns even if it's empty
    real_df = pd.DataFrame(real_rows, columns=real_df_cols)

    # 4) Promedio vendor + dow: para missing block
    vendor_dow_imp = summarize(
        [vendor_col, dow_num_col, dow_name_col],
        suffix='vendor_dow'
    )

    # 5) Promedio vendor + block: para missing dow completo
    vendor_block_imp = summarize(
        [vendor_col, block_col],
        suffix='vendor_block'
    )

    # 6) Promedio vendor completo: fallback final
    vendor_imp = summarize(
        [vendor_col],
        suffix='vendor'
    )

    # 7) Merge de todo
    out = (
        grid
        .merge(real_df, on=group_keys, how='left')
        .merge(vendor_meta, on=vendor_col, how='left')
        .merge(vendor_dow_imp, on=[vendor_col, dow_num_col, dow_name_col], how='left')
        .merge(vendor_block_imp, on=[vendor_col, block_col], how='left')
        .merge(vendor_imp, on=vendor_col, how='left')
    )

    # 8) Imputar métricas con prioridad:
    # real > vendor+dow > vendor+block > vendor
    for col in metric_cols:
        out[col] = (
            out[col]
            .fillna(out[f'{col}_vendor_dow'])
            .fillna(out[f'{col}_vendor_block'])
            .fillna(out[f'{col}_vendor'])
        )

    # 9) Filas artificiales no tienen orders reales
    for col in count_cols:
        out[col] = out[col].fillna(0)

    # 10) Flag de auditoría
    out['imputation_level'] = np.select(
        [
            out['_is_real_row'].eq(1),
            out[f'{metric_cols[0]}_vendor_dow'].notna(),
            out[f'{metric_cols[0]}_vendor_block'].notna(),
            out[f'{metric_cols[0]}_vendor'].notna()
        ],
        [
            'real',
            'vendor_dow_avg',
            'vendor_block_avg',
            'vendor_avg'
        ],
        default='no_data'
    )

    # 11) Limpiar auxiliares
    aux_cols = ['_is_real_row']

    for col in metric_cols:
        aux_cols += [
            f'{col}_vendor_dow',
            f'{col}_vendor_block',
            f'{col}_vendor'
        ]

    out = out.drop(columns=[c for c in aux_cols if c in out.columns])

    # 12) Orden final
    final_cols = [
        'franchise_id', 'franchise_name', 'city_name', 'vertical_type',
        'franchise_orders', 'franchise_stores',
        'vendor_code', 'store_name', 'store_orders_total',
        'day_of_week_num', 'day_of_week_name', 'time_block',
        'total_orders', 'slow_orders', 'non_seamless_orders',
        'ept', 'awt', 'tt', 'ctp',
        'imputation_level'
    ]

    final_cols = [c for c in final_cols if c in out.columns]

    out = (
        out[final_cols]
        .sort_values(
            ['franchise_name', 'store_name', 'day_of_week_num', 'time_block']
        )
        .reset_index(drop=True)
    )


    # Auditoría final
    n_vendors = out['vendor_code'].nunique()
    expected_rows = n_vendors * 7 * len(target_time_blocks)

    print(f'Vendors procesados: {n_vendors}')
    print(f'Filas generadas: {len(out)} de {expected_rows} esperadas')
    print(f'Filas con EPT NaN: {out["ept"].isna().sum()}')

    print('\nImputation levels:')
    print(out['imputation_level'].value_counts(dropna=False))

    vendors_no_data = out.loc[
        out['ept'].isna(),
        'vendor_code'
    ].drop_duplicates().tolist()

    if vendors_no_data:
        print(f'\nVendors sin EPT utilizable: {vendors_no_data}')
    else:
        print('\nTodos los vendors tienen EPT.')

    duplicates = out.duplicated(
        ['vendor_code', 'day_of_week_num', 'time_block']
    ).sum()

    print(f'Duplicados vendor+dow+block: {duplicates}')

    return out

# Calculo new EPTs

### calcular factor de EPT

In [22]:
# Normalizar IDs también en la tabla agregada.
df_grouped["vendor_code"] = normalize_id(df_grouped["vendor_code"])
df_grouped["franchise_id"] = normalize_id(df_grouped["franchise_id"])

rule_metadata = ["ept_new", "wave", "flag", "_input_row_number"]

# Las reglas de franquicia solo son filas sin vendor_code.
franchise_rules = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("franchise"),
        ["franchise_id"] + rule_metadata
    ]
    .drop_duplicates("franchise_id")
    .rename(columns={
        "ept_new": "ept_new_franchise",
        "wave": "wave_franchise",
        "flag": "flag_franchise",
        "_input_row_number": "input_row_number_franchise"
    })
)

# Toda fila con vendor_code es individual y tiene prioridad sobre la franquicia.
vendor_rules = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("vendor"),
        ["vendor_code"] + rule_metadata
    ]
    .drop_duplicates("vendor_code")
    .rename(columns={
        "ept_new": "ept_new_vendor",
        "wave": "wave_vendor",
        "flag": "flag_vendor",
        "_input_row_number": "input_row_number_vendor"
    })
)

df_reduction = (
    df_grouped
    .merge(franchise_rules, on="franchise_id", how="left")
    .merge(vendor_rules, on="vendor_code", how="left")
)

vendor_rule_applies = df_reduction["ept_new_vendor"].notna()
franchise_rule_applies = (
    ~vendor_rule_applies
    & df_reduction["ept_new_franchise"].notna()
)

df_reduction["ept_new"] = (
    df_reduction["ept_new_vendor"]
    .combine_first(df_reduction["ept_new_franchise"])
)
df_reduction["wave"] = df_reduction["wave_vendor"].where(
    vendor_rule_applies,
    df_reduction["wave_franchise"]
)
df_reduction["flag"] = df_reduction["flag_vendor"].where(
    vendor_rule_applies,
    df_reduction["flag_franchise"]
)
df_reduction["input_row_number"] = df_reduction[
    "input_row_number_vendor"
].where(
    vendor_rule_applies,
    df_reduction["input_row_number_franchise"]
)

df_reduction["adjustment_scope"] = np.select(
    [vendor_rule_applies, franchise_rule_applies],
    ["vendor", "franchise"],
    default="not_targeted"
)
df_reduction["source_franchise_id"] = df_reduction[
    "franchise_id"
].where(df_reduction["adjustment_scope"].eq("franchise"))

has_wave = (
    df_reduction["wave"].astype("string").str.strip().ne("").fillna(False)
)
df_reduction["adjustment_origin"] = np.select(
    [
        df_reduction["adjustment_scope"].eq("not_targeted"),
        has_wave
    ],
    ["not_targeted", "wave"],
    default="manual"
)

# Diferencia entre EPT actual promedio y EPT objetivo solicitado.
df_reduction["ept_diff"] = df_reduction["ept"] - df_reduction["ept_new"]
is_targeted = df_reduction["ept_new"].notna()

if reductions_only:
    df_reduction["apply_change"] = (
        is_targeted
        & (
            df_reduction["ept"].isna()
            | (df_reduction["ept_diff"] >= required_diff_mins)
        )
    )
else:
    df_reduction["apply_change"] = (
        is_targeted
        & (
            df_reduction["ept"].isna()
            | (df_reduction["ept_diff"].abs() >= required_diff_mins)
        )
    )

# Porcentaje que después aplica aplicar_factor_ept().
df_reduction["ept_reduction_pct"] = np.where(
    df_reduction["apply_change"],
    1 - (df_reduction["ept_new"] / df_reduction["ept"]),
    0
)

df_reduction[[
    "franchise_id",
    "vertical_type",
    "vendor_code",
    "adjustment_origin",
    "adjustment_scope",
    "wave",
    "flag",
    "ept",
    "ept_new",
    "ept_diff",
    "apply_change",
    "ept_reduction_pct"
]].head()


,franchise_id,vertical_type,vendor_code,adjustment_origin,adjustment_scope,wave,flag,ept,ept_new,ept_diff,apply_change,ept_reduction_pct
0,0016900002xQv80AAC,restaurants,519467,manual,vendor,<NA>,reajuste regional,11.634839,18.0,-6.365161,True,-0.547078
1,0011r00002VoHw6AAF,restaurants,535773,manual,vendor,<NA>,reajuste regional,27.230030,25.0,2.230030,False,0.000000
2,0011r00002VoHw6AAF,restaurants,341835,manual,vendor,<NA>,reajuste regional,28.518836,25.0,3.518836,True,0.123386
3,0011r00002VoHw6AAF,restaurants,133603,manual,vendor,<NA>,reajuste regional,27.058330,25.0,2.058330,False,0.000000
4,0011r00002VoHw6AAF,restaurants,133509,manual,vendor,<NA>,reajuste regional,32.213577,25.0,7.213577,True,0.223930


### parchar dow-blocks faltantes con avg

In [23]:
# Procesar únicamente vendors resueltos por alguna regla vigente.
target_vendor_codes = set(
    df_reduction.loc[df_reduction["apply_change"]==True, "vendor_code"]
    .dropna()
)

ept_actuales_targeted = ept_actuales.loc[
    ept_actuales["vendor_code"].isin(target_vendor_codes)
].copy()

ept_actuales_mod_patched = rellenar_vendor_dow_block(ept_actuales_targeted)

Vendors procesados: 7
Filas generadas: 147 de 147 esperadas
Filas con EPT NaN: 0

Imputation levels:
imputation_level
real                136
vendor_dow_avg        8
vendor_block_avg      3
Name: count, dtype: int64

Todos los vendors tienen EPT.
Duplicados vendor+dow+block: 0


### aplicar factor EPT

In [24]:
ept_actuales_new_ept = aplicar_factor_ept(
    ept_actuales_mod_patched,
    df_reduction
)

print(
    f"Stores a modificar: "
    f"{df_reduction.loc[df_reduction['apply_change'], 'vendor_code'].nunique()}"
)

print(
    f"De un total objetivo de: "
    f"{df_reduction.loc[df_reduction['ept_new'].notna(), 'vendor_code'].nunique()}"
)

origin_summary = (
    df_reduction.loc[
        df_reduction["ept_new"].notna(),
        ["vendor_code", "adjustment_origin", "adjustment_scope"]
    ]
    .groupby(
        ["adjustment_origin", "adjustment_scope"],
        dropna=False,
        as_index=False
    )
    .agg(vendors=("vendor_code", "nunique"))
)

print("\nOrigen y alcance del EPT objetivo:")
print(origin_summary.to_string(index=False))


Avg EPT actual: 30.03
Avg EPT nuevo:  25.78
Delta avg EPT:  -4.25
Stores a modificar: 7
De un total objetivo de: 11

Origen y alcance del EPT objetivo:
adjustment_origin adjustment_scope  vendors
           manual           vendor       11


In [25]:
# Vendors que tienen al menos un EPT histórico numérico válido.
ept_history_numeric = pd.to_numeric(ept_actuales["ept"], errors="coerce")
vendors_with_history = set(
    ept_actuales.loc[ept_history_numeric.notna(), "vendor_code"].dropna()
)

# Targets ya expandidos por la query: incluyen los vendors de cada franquicia.
target_meta_cols = [
    "vendor_code", "franchise_id", "franchise_name",
    "city_name", "vertical_type", "store_name", "ept_new",
    "adjustment_origin", "adjustment_scope", "source_franchise_id",
    "wave", "flag", "input_row_number", "apply_change"
]
target_meta_cols = [
    column for column in target_meta_cols
    if column in df_reduction.columns
]

resolved_targets = (
    df_reduction.loc[df_reduction["ept_new"].notna(), target_meta_cols]
    .drop_duplicates("vendor_code")
    .copy()
)

# Fallback para un vendor explícito que no apareció en la query.
unresolved_explicit = vendor_rules.rename(columns={
    "ept_new_vendor": "ept_new",
    "wave_vendor": "wave",
    "flag_vendor": "flag",
    "input_row_number_vendor": "input_row_number"
}).copy()

for column in [
    "franchise_id", "franchise_name", "city_name",
    "vertical_type", "store_name"
]:
    unresolved_explicit[column] = pd.NA

unresolved_explicit["adjustment_scope"] = "vendor"
unresolved_explicit["source_franchise_id"] = pd.NA
unresolved_explicit["adjustment_origin"] = np.where(
    unresolved_explicit["wave"].astype("string").str.strip().ne("").fillna(False),
    "wave",
    "manual"
)
unresolved_explicit["apply_change"] = True
unresolved_explicit = unresolved_explicit.reindex(columns=target_meta_cols)

all_targets = (
    pd.concat(
        [resolved_targets, unresolved_explicit],
        ignore_index=True,
        sort=False
    )
    .drop_duplicates("vendor_code", keep="first")
)

missing_vendors = all_targets.loc[
    ~all_targets["vendor_code"].isin(vendors_with_history)
].copy()

# Quitar sus filas actuales con EPT NaN para evitar duplicados.
ept_actuales_new_ept = ept_actuales_new_ept.loc[
    ~ept_actuales_new_ept["vendor_code"].astype(str).isin(
        missing_vendors["vendor_code"].astype(str)
    )
].copy()

# Crear las 21 combinaciones: 7 días x 3 bloques.
days = pd.DataFrame({
    "day_of_week_num": range(1, 8),
    "day_of_week_name": [
        "Sunday", "Monday", "Tuesday", "Wednesday",
        "Thursday", "Friday", "Saturday"
    ]
})

blocks = pd.DataFrame({
    "time_block": ["lunch", "dinner", "valle"]
})

missing_rows = (
    missing_vendors
    .merge(days, how="cross")
    .merge(blocks, how="cross")
)

# Sin historia, usar directamente el EPT objetivo solicitado.
missing_rows["ept"] = missing_rows["ept_new"]
missing_rows["ept_delta_min"] = 0
missing_rows["total_orders"] = 1
missing_rows["imputation_level"] = "new_preps_no_history"

ept_actuales_new_ept = pd.concat(
    [ept_actuales_new_ept, missing_rows],
    ignore_index=True,
    sort=False
)

# Los vendors explícitos sin match también deben llegar al template TES.
target_vendor_codes.update(
    missing_vendors["vendor_code"].dropna().astype(str).tolist()
)

print(
    f"Vendors sin historia agregados: "
    f"{missing_vendors['vendor_code'].nunique()}"
)


Vendors sin historia agregados: 0


### (check)

In [26]:
# checkear random vendors
random_vendor_code = ept_actuales_new_ept['vendor_code'].sample(1).iloc[0]
ept_actuales_new_ept[ept_actuales_new_ept['vendor_code'] == random_vendor_code][[
    'franchise_name',
    'vendor_code',
    'day_of_week_name',
    'time_block',
    'ept',
    'ept_new',
    'ept_delta_min',
    # 'ept_reduction_pct'
]].round(2)

,franchise_name,vendor_code,day_of_week_name,time_block,ept,ept_new,ept_delta_min
105,NaN,418988,Sunday,dinner,36.38,26.97,-9.41
106,NaN,418988,Sunday,lunch,32.00,23.72,-8.28
107,NaN,418988,Sunday,valle,38.77,28.74,-10.03
108,NaN,418988,Monday,dinner,36.85,27.32,-9.53
109,NaN,418988,Monday,lunch,35.27,26.15,-9.13
110,NaN,418988,Monday,valle,36.97,27.41,-9.57
111,NaN,418988,Tuesday,dinner,34.67,25.70,-8.97
112,NaN,418988,Tuesday,lunch,28.00,20.76,-7.24
113,NaN,418988,Tuesday,valle,32.00,23.72,-8.28
114,NaN,418988,Wednesday,dinner,28.00,20.76,-7.24


### capping

In [27]:
random_vendor_code = ept_actuales_new_ept['vendor_code'].sample(1).iloc[0]

ept_actuales_new_ept_capped = median_cap(
    ept_actuales_new_ept,
    multiplier=0.8
)
ept_actuales_new_ept_capped[ept_actuales_new_ept_capped['vendor_code'] == random_vendor_code][[
    'franchise_name',
    'vendor_code',
    'day_of_week_name',
    'time_block',
    'ept',
    'ept_new',
    'ept_delta_min',
    'upper_limit',
    'is_capped'
    # 'ept_reduction_pct'
]].round(2)

# ept_actuales_new_ept_capped[ept_actuales_new_ept_capped['vendor_code'] == random_vendor_code]

Number of rows capped: 135


,franchise_name,vendor_code,day_of_week_name,time_block,ept,ept_new,ept_delta_min,upper_limit,is_capped
42,Papa John's,133509,Sunday,dinner,45.06,20.85,-10.09,20.85,True
43,Papa John's,133509,Sunday,lunch,33.58,20.85,-7.52,20.85,True
44,Papa John's,133509,Sunday,valle,31.35,20.85,-7.02,20.85,True
45,Papa John's,133509,Monday,dinner,34.06,20.85,-7.63,20.85,True
46,Papa John's,133509,Monday,lunch,28.61,20.85,-6.41,20.85,True
47,Papa John's,133509,Monday,valle,26.13,20.28,-5.85,20.85,False
48,Papa John's,133509,Tuesday,dinner,23.62,18.33,-5.29,20.85,False
49,Papa John's,133509,Tuesday,lunch,24.08,18.69,-5.39,20.85,False
50,Papa John's,133509,Tuesday,valle,23.15,17.97,-5.18,20.85,False
51,Papa John's,133509,Wednesday,dinner,41.60,20.85,-9.32,20.85,True


### formato TES y exportación

In [28]:
# ept_actuales_mod_TES = exportar_template_ops(df_manual_modified)
ept_actuales_mod_TES = exportar_template_ops(
    ept_actuales_new_ept_capped[
        ept_actuales_new_ept_capped["vendor_code"].isin(target_vendor_codes)
    ]
)
ept_actuales
# resultados en: https://drive.google.com/drive/u/0/folders/1eRW_EDJ0hy687Ns1RZddRHJoXTTDFzmA

Filas totales exportadas: 245
Archivos creados: 1
Carpeta creada: c:\Users\pablo.villanueva\.gemini\antigravity\scratch\bigquery_pipeline\github primer repo\lower-ept-y-awt\results\template_ops_tiempos_20260910_173408_962642
c:\Users\pablo.villanueva\.gemini\antigravity\scratch\bigquery_pipeline\github primer repo\lower-ept-y-awt\results\template_ops_tiempos_20260910_173408_962642\part_01.csv


,franchise_id,franchise_name,city_name,vertical_type,franchise_orders,franchise_stores,franchise_orders_with_fir,franchise_fir,vendor_code,store_name,store_orders_total,store_orders_with_fir,store_fir,day_of_week_num,day_of_week_name,time_block,total_orders,slow_orders,non_seamless_orders,hd_total_orders,hd_total_minutes_added,started_triggers,avg_trigger_duration_min,orders_with_fir,fir_ratio,ept,awt,tt,ctp,fir
0,<NA>,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,lunch,14,0,4,0,0.0,0,NaN,14,1.0,8.29,1.00,9.29,11.43,187.76
1,<NA>,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,7.00,0.00,7.00,7.00,184.25
2,<NA>,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,2,Monday,valle,16,0,0,0,0.0,0,NaN,16,1.0,8.50,6.12,14.62,16.56,192.74
3,<NA>,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,3,Tuesday,lunch,15,1,4,0,0.0,0,NaN,15,1.0,9.07,3.52,12.58,19.40,190.89
4,<NA>,NaN,Rancagua,restaurants,156,1,155,0.9936,301949,Gohan Club,156,155,0.9936,3,Tuesday,dinner,5,0,0,0,0.0,0,NaN,5,1.0,9.20,0.12,9.32,8.60,187.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,0011r00002VoHw6AAF,Papa John's,Concepcion,restaurants,2612,5,0,0.0000,291016,Papa John´s - Coronel,605,0,0.0000,6,Friday,dinner,37,27,27,21,210.0,4,815.00,0,0.0,47.84,4.21,52.05,55.19,NaN
201,0011r00002VoHw6AAF,Papa John's,Concepcion,restaurants,2612,5,0,0.0000,291016,Papa John´s - Coronel,605,0,0.0000,6,Friday,valle,44,10,13,21,210.0,9,785.44,0,0.0,36.02,2.42,38.44,40.75,NaN
202,0011r00002VoHw6AAF,Papa John's,Concepcion,restaurants,2612,5,0,0.0000,291016,Papa John´s - Coronel,605,0,0.0000,7,Saturday,lunch,25,5,8,6,60.0,0,NaN,0,0.0,34.44,4.75,39.19,41.04,NaN
203,0011r00002VoHw6AAF,Papa John's,Concepcion,restaurants,2612,5,0,0.0000,291016,Papa John´s - Coronel,605,0,0.0000,7,Saturday,dinner,37,21,22,20,200.0,4,273.25,0,0.0,45.59,5.31,50.90,52.70,NaN


### auditoría y tabla estandarizada `results_export`

Convención de signos del output:

* `ept_change_min = ept_new - ept_old`.
* `ept_change_pct = ept_new / ept_old - 1`.
* Una reducción queda negativa y un alza queda positiva en ambas columnas.

`results_export` funciona como staging de la ejecución actual y se reemplaza en cada corrida. Para el historial consolidado en el otro spreadsheet, el nombre recomendado es `ept_adjustment_history`.


In [29]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

# Este bloque debe ejecutarse después de crear ept_actuales_new_ept_capped.
summary_base = ept_actuales_new_ept_capped.copy()

numeric_cols = [
    "ept", "ept_new", "awt", "total_orders",
    "slow_orders", "non_seamless_orders"
]
for col in numeric_cols:
    if col not in summary_base.columns:
        summary_base[col] = np.nan
    summary_base[col] = pd.to_numeric(summary_base[col], errors="coerce")

# Las filas creadas para vendors sin historia traen total_orders=1 solo para
# permitir generar la grilla TES. No deben contarse como órdenes reales.
summary_base["has_real_history"] = (
    summary_base["ept"].notna()
    & summary_base["total_orders"].gt(0)
    & ~summary_base["imputation_level"].eq("new_preps_no_history")
)
summary_base["orders_for_summary"] = np.where(
    summary_base["has_real_history"],
    summary_base["total_orders"],
    0
)

# Es el valor que realmente se manda en PREPARATION-TIME, porque el
# exportador TES usa astype(int). Para EPT positivos equivale a truncar.
summary_base["ept_new_tes"] = np.trunc(summary_base["ept_new"])
summary_base["old_ept_minutes"] = (
    summary_base["ept"] * summary_base["orders_for_summary"]
)
summary_base["new_ept_minutes"] = (
    summary_base["ept_new"] * summary_base["orders_for_summary"]
)
summary_base["new_ept_tes_minutes"] = (
    summary_base["ept_new_tes"] * summary_base["orders_for_summary"]
)
summary_base["awt_minutes"] = (
    summary_base["awt"] * summary_base["orders_for_summary"]
)

def first_notna(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan

# Metadata de la instrucción ganadora: una fila por vendor.
rule_cols = [
    "vendor_code", "ept_new", "adjustment_origin", "adjustment_scope",
    "source_franchise_id", "wave", "flag", "input_row_number",
    "apply_change"
]
rule_cols = [column for column in rule_cols if column in df_reduction.columns]
rule_meta_resolved = df_reduction[rule_cols].drop_duplicates("vendor_code")

# Los vendor_code explícitos sin historia no aparecen en df_reduction, pero
# sí se agregaron a la grilla final. Incorporarlos con la misma metadata.
missing_rule_cols = [
    "vendor_code", "ept_new", "adjustment_origin", "adjustment_scope",
    "source_franchise_id", "wave", "flag", "input_row_number",
    "apply_change"
]
missing_rule_cols = [
    column for column in missing_rule_cols
    if column in missing_vendors.columns
]
rule_meta_missing = missing_vendors[missing_rule_cols].copy()
rule_meta_missing["apply_change"] = True

rule_meta = (
    pd.concat([rule_meta_resolved, rule_meta_missing], ignore_index=True)
    .drop_duplicates("vendor_code", keep="first")
    .rename(columns={"ept_new": "ept_target_requested"})
)

vendor_rows = []
for vendor_code, g in summary_base.groupby("vendor_code", dropna=False):
    orders = g["orders_for_summary"].sum()

    if orders > 0:
        ept_old = g["old_ept_minutes"].sum() / orders
        ept_new = g["new_ept_minutes"].sum() / orders
        ept_new_tes = g["new_ept_tes_minutes"].sum() / orders
        awt_old = g["awt_minutes"].sum(min_count=1) / orders
        slow_orders = g.loc[g["has_real_history"], "slow_orders"].sum(min_count=1)
        non_seamless_orders = g.loc[
            g["has_real_history"], "non_seamless_orders"
        ].sum(min_count=1)
    else:
        ept_old = np.nan
        ept_new = g["ept_new"].mean()
        ept_new_tes = g["ept_new_tes"].mean()
        awt_old = np.nan
        slow_orders = np.nan
        non_seamless_orders = np.nan

    reduction_min = ept_old - ept_new if pd.notna(ept_old) else np.nan
    reduction_pct = (
        reduction_min / ept_old
        if pd.notna(ept_old) and ept_old != 0
        else np.nan
    )
    reduction_tes_min = (
        ept_old - ept_new_tes if pd.notna(ept_old) else np.nan
    )

    vendor_rows.append({
        "franchise_id": first_notna(g["franchise_id"]),
        "franchise_name": first_notna(g["franchise_name"]),
        "vendor_code": vendor_code,
        "store_name": first_notna(g["store_name"]),
        "city_name": first_notna(g["city_name"]),
        "vertical_type": first_notna(g["vertical_type"]),
        "has_history": orders > 0,
        "orders_7d": orders,
        "ept_old": ept_old,
        "ept_new": ept_new,
        "reduction_min": reduction_min,
        "reduction_pct": reduction_pct,
        "ept_new_tes": ept_new_tes,
        "reduction_tes_min": reduction_tes_min,
        "estimated_minutes_reduced_7d": (
            reduction_tes_min * orders
            if pd.notna(reduction_tes_min)
            else np.nan
        ),
        "awt_old": awt_old,
        "slow_share": (
            slow_orders / orders if orders > 0 and pd.notna(slow_orders) else np.nan
        ),
        "non_seamless_share": (
            non_seamless_orders / orders
            if orders > 0 and pd.notna(non_seamless_orders)
            else np.nan
        ),
        "imputed_blocks": int((g["imputation_level"] != "real").sum()),
        "capped_blocks": int(g["is_capped"].fillna(False).sum())
    })

vendor_summary = pd.DataFrame(vendor_rows).merge(
    rule_meta, on="vendor_code", how="left"
)
vendor_summary["apply_change"] = vendor_summary["apply_change"].fillna(False)
vendor_summary["change_type"] = np.select(
    [
        ~vendor_summary["has_history"],
        vendor_summary["apply_change"] & vendor_summary["reduction_tes_min"].gt(0),
        vendor_summary["apply_change"] & vendor_summary["reduction_tes_min"].lt(0)
    ],
    ["no_history", "reduction", "increase"],
    default="unchanged"
)

# Una tienda sin franchise_id constituye su propio grupo; así no se mezclan
# todos los independientes bajo un franchise NULL.
vendor_summary["franchise_group_id"] = np.where(
    vendor_summary["franchise_id"].notna(),
    vendor_summary["franchise_id"].astype("string"),
    "VENDOR_" + vendor_summary["vendor_code"].astype("string")
)
vendor_summary["franchise_group_name"] = vendor_summary["franchise_name"].fillna(
    "Independent / " + vendor_summary["store_name"].fillna(
        vendor_summary["vendor_code"].astype("string")
    )
)

franchise_rows = []
for group_id, g in vendor_summary.groupby("franchise_group_id", dropna=False):
    history = g["has_history"] & g["orders_7d"].gt(0)
    orders = g.loc[history, "orders_7d"].sum()

    if orders > 0:
        ept_old = np.average(
            g.loc[history, "ept_old"],
            weights=g.loc[history, "orders_7d"]
        )
        ept_new = np.average(
            g.loc[history, "ept_new"],
            weights=g.loc[history, "orders_7d"]
        )
        ept_new_tes = np.average(
            g.loc[history, "ept_new_tes"],
            weights=g.loc[history, "orders_7d"]
        )
        awt_old = np.average(
            g.loc[history & g["awt_old"].notna(), "awt_old"],
            weights=g.loc[history & g["awt_old"].notna(), "orders_7d"]
        ) if (history & g["awt_old"].notna()).any() else np.nan
    else:
        ept_old = np.nan
        ept_new = g["ept_new"].mean()
        ept_new_tes = g["ept_new_tes"].mean()
        awt_old = np.nan

    reduction_min = ept_old - ept_new if pd.notna(ept_old) else np.nan
    reduction_tes_min = (
        ept_old - ept_new_tes if pd.notna(ept_old) else np.nan
    )

    franchise_rows.append({
        "franchise_group_id": group_id,
        "franchise_id": first_notna(g["franchise_id"]),
        "franchise_name": first_notna(g["franchise_group_name"]),
        "vertical_type": ", ".join(
            sorted(g["vertical_type"].dropna().astype(str).unique())
        ),
        "vendors_exported": g["vendor_code"].nunique(),
        "vendors_affected": g.loc[g["apply_change"], "vendor_code"].nunique(),
        "vendors_reduced": g.loc[g["change_type"].eq("reduction"), "vendor_code"].nunique(),
        "vendors_increased": g.loc[g["change_type"].eq("increase"), "vendor_code"].nunique(),
        "vendors_unchanged": g.loc[g["change_type"].eq("unchanged"), "vendor_code"].nunique(),
        "vendors_without_history": g.loc[~g["has_history"], "vendor_code"].nunique(),
        "orders_7d": orders,
        "ept_old": ept_old,
        "ept_new": ept_new,
        "reduction_min": reduction_min,
        "reduction_pct": (
            reduction_min / ept_old
            if pd.notna(ept_old) and ept_old != 0
            else np.nan
        ),
        "ept_new_tes": ept_new_tes,
        "reduction_tes_min": reduction_tes_min,
        "estimated_minutes_reduced_7d": g["estimated_minutes_reduced_7d"].sum(min_count=1),
        "awt_old": awt_old,
        "slow_share": (
            np.average(
                g.loc[history & g["slow_share"].notna(), "slow_share"],
                weights=g.loc[history & g["slow_share"].notna(), "orders_7d"]
            ) if (history & g["slow_share"].notna()).any() else np.nan
        ),
        "non_seamless_share": (
            np.average(
                g.loc[history & g["non_seamless_share"].notna(), "non_seamless_share"],
                weights=g.loc[history & g["non_seamless_share"].notna(), "orders_7d"]
            ) if (history & g["non_seamless_share"].notna()).any() else np.nan
        )
    })

franchise_summary = pd.DataFrame(franchise_rows)

# Redondear solo para la salida; los cálculos anteriores mantienen precisión.
round_cols = [
    "ept_old", "ept_new", "reduction_min", "reduction_pct",
    "ept_new_tes", "reduction_tes_min",
    "estimated_minutes_reduced_7d", "awt_old",
    "slow_share", "non_seamless_share"
]
for df_out in [vendor_summary, franchise_summary]:
    cols = [c for c in round_cols if c in df_out.columns]
    df_out[cols] = df_out[cols].round(4)

vendor_summary = vendor_summary.sort_values(
    ["franchise_group_name", "store_name", "vendor_code"]
).reset_index(drop=True)
franchise_summary = franchise_summary.sort_values(
    ["franchise_name", "franchise_group_id"]
).reset_index(drop=True)

if os.path.exists('/content/drive/MyDrive'):
    dest_folder = '/content/drive/MyDrive/lower ept y awt/results'
else:
    dest_folder = os.path.join(os.getcwd(), 'results')
os.makedirs(dest_folder, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
vendor_path = os.path.join(
    dest_folder, f"summary_reducciones_vendor_{timestamp}.csv"
)
franchise_path = os.path.join(
    dest_folder, f"summary_reducciones_franchise_{timestamp}.csv"
)

vendor_summary.to_csv(vendor_path, index=False)
franchise_summary.to_csv(franchise_path, index=False)

print(f"CSV vendor: {vendor_path}")
print(f"CSV franchise: {franchise_path}")
print(
    f"Vendors exportados: {vendor_summary['vendor_code'].nunique():,} | "
    f"Vendors afectados: {vendor_summary['apply_change'].sum():,} | "
    f"Sin historia: {(~vendor_summary['has_history']).sum():,}"
)

display(franchise_summary.head(20))
display(vendor_summary.head(20))

CSV vendor: c:\Users\pablo.villanueva\.gemini\antigravity\scratch\bigquery_pipeline\github primer repo\lower-ept-y-awt\results\summary_reducciones_vendor_20260910_173409.csv
CSV franchise: c:\Users\pablo.villanueva\.gemini\antigravity\scratch\bigquery_pipeline\github primer repo\lower-ept-y-awt\results\summary_reducciones_franchise_20260910_173409.csv
Vendors exportados: 7 | Vendors afectados: 7 | Sin historia: 0


,franchise_group_id,franchise_id,franchise_name,vertical_type,vendors_exported,vendors_affected,vendors_reduced,vendors_increased,vendors_unchanged,vendors_without_history,orders_7d,ept_old,ept_new,reduction_min,reduction_pct,ept_new_tes,reduction_tes_min,estimated_minutes_reduced_7d,awt_old,slow_share,non_seamless_share
0,0016900002xQv80AAC,0016900002xQv80AAC,Club Del Poke - LATAM,restaurants,1,1,0,1,0,0,93.0,13.3553,16.7379,-3.3826,-0.2533,15.9570,-2.6017,-241.96,11.6838,0.1183,0.3226
1,VENDOR_286542,NaN,Independent / Mi Delizie Sur Sushi,restaurants,1,1,1,0,0,0,701.0,31.9497,20.7754,11.1743,0.3497,20.0000,11.9497,8376.77,1.7734,0.3937,0.4151
2,VENDOR_418988,NaN,Independent / Nori & Nigiri,restaurants,1,1,1,0,0,0,85.0,36.7054,22.3254,14.3800,0.3918,21.9529,14.7525,1253.96,4.6588,0.2824,0.3765
3,VENDOR_229581,NaN,Independent / Sushi Camba Antofagasta Norte,restaurants,1,1,1,0,0,0,59.0,38.2888,26.5773,11.7115,0.3059,26.0000,12.2888,725.04,5.0637,0.3390,0.4237
4,0011r00002VoHw6AAF,0011r00002VoHw6AAF,Papa John's,restaurants,3,3,3,0,0,0,1798.0,32.8896,20.3602,12.5294,0.3810,19.9549,12.9346,23256.46,3.1944,0.2264,0.2725


,franchise_id,franchise_name,vendor_code,store_name,city_name,vertical_type,has_history,orders_7d,ept_old,ept_new,reduction_min,reduction_pct,ept_new_tes,reduction_tes_min,estimated_minutes_reduced_7d,awt_old,slow_share,non_seamless_share,imputed_blocks,capped_blocks,ept_target_requested,adjustment_origin,adjustment_scope,source_franchise_id,wave,flag,input_row_number,apply_change,change_type,franchise_group_id,franchise_group_name
0,0016900002xQv80AAC,Club Del Poke - LATAM,519467,Club Del Poke - Copiapó.,Copiapo,restaurants,True,93.0,13.3553,16.7379,-3.3826,-0.2533,15.9570,-2.6017,-241.96,11.6838,0.1183,0.3226,1,20,18.0,manual,vendor,<NA>,<NA>,reajuste regional,12,True,increase,0016900002xQv80AAC,Club Del Poke - LATAM
1,NaN,NaN,286542,Mi Delizie Sur Sushi,Antofagasta,restaurants,True,701.0,31.9497,20.7754,11.1743,0.3497,20.0000,11.9497,8376.77,1.7734,0.3937,0.4151,1,21,26.0,manual,vendor,<NA>,<NA>,reajuste regional,7,True,reduction,VENDOR_286542,Independent / Mi Delizie Sur Sushi
2,NaN,NaN,418988,Nori & Nigiri,Concepcion,restaurants,True,85.0,36.7054,22.3254,14.3800,0.3918,21.9529,14.7525,1253.96,4.6588,0.2824,0.3765,4,19,27.0,manual,vendor,<NA>,<NA>,reajuste regional,8,True,reduction,VENDOR_418988,Independent / Nori & Nigiri
3,NaN,NaN,229581,Sushi Camba Antofagasta Norte,Antofagasta,restaurants,True,59.0,38.2888,26.5773,11.7115,0.3059,26.0000,12.2888,725.04,5.0637,0.3390,0.4237,4,21,33.0,manual,vendor,<NA>,<NA>,reajuste regional,9,True,reduction,VENDOR_229581,Independent / Sushi Camba Antofagasta Norte
4,0011r00002VoHw6AAF,Papa John's,341835,Papa Johns Pizza - Copiapo,Copiapo,restaurants,True,550.0,28.5188,19.0644,9.4544,0.3315,18.9709,9.5479,5251.36,2.9497,0.1400,0.1909,0,20,25.0,manual,vendor,<NA>,<NA>,reajuste regional,4,True,reduction,0011r00002VoHw6AAF,Papa John's
5,0011r00002VoHw6AAF,Papa John's,133509,Papa Johns Pizza - Punta Arenas,Punta arenas,restaurants,True,643.0,35.3240,20.7073,14.6167,0.4138,19.8725,15.4515,9935.33,3.6028,0.3188,0.3515,1,16,25.0,manual,vendor,<NA>,<NA>,reajuste regional,3,True,reduction,0011r00002VoHw6AAF,Papa John's
6,0011r00002VoHw6AAF,Papa John's,291016,Papa John´s - Coronel,Concepcion,restaurants,True,605.0,34.2757,21.1691,13.1065,0.3824,20.9372,13.3385,8069.77,2.9829,0.2066,0.2628,0,18,25.0,manual,vendor,<NA>,<NA>,reajuste regional,2,True,reduction,0011r00002VoHw6AAF,Papa John's


In [30]:
from IPython import display
# ============================================================
# ESCRIBIR results_export Y MARCAR to_adjust_now
# ============================================================

import numpy as np
import pandas as pd
import gspread

# Una fila por vendor efectivamente incluido en el template TES.
export_audit = vendor_summary.loc[
    vendor_summary["apply_change"].fillna(False)
].copy()

# ept_new conserva el objetivo solicitado en to_adjust_now.
export_audit["ept_new"] = export_audit[
    "ept_target_requested"
].combine_first(export_audit["ept_new"])

# Convención: nuevo - antiguo. Reducciones negativas; alzas positivas.
export_audit["ept_change_min"] = (
    export_audit["ept_new"] - export_audit["ept_old"]
)
export_audit["ept_change_pct"] = np.where(
    export_audit["ept_old"].notna()
    & export_audit["ept_old"].ne(0),
    (export_audit["ept_new"] / export_audit["ept_old"]) - 1,
    np.nan
)
export_audit["executed_at"] = executed_at

# Esquema final mínimo para el paste y el historial consolidado.
result_columns = [
    "executed_at",
    "wave",
    "flag",
    "adjustment_scope",
    "franchise_id",
    "franchise_name",
    "vertical_type",
    "vendor_code",
    "store_name",
    "ept_old",
    "ept_change_pct",
    "ept_new",
    "ept_change_min"
]

for column in result_columns:
    if column not in export_audit.columns:
        export_audit[column] = pd.NA

results_export = export_audit[result_columns].copy()

for column in ["ept_old", "ept_new", "ept_change_min"]:
    results_export[column] = pd.to_numeric(
        results_export[column], errors="coerce"
    ).round(1)

results_export["ept_change_pct"] = pd.to_numeric(
    results_export["ept_change_pct"], errors="coerce"
).round(4)

results_export = results_export.sort_values(
    ["wave", "franchise_name", "store_name", "vendor_code"],
    na_position="last"
).reset_index(drop=True)


def clean_sheet_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


result_values = [result_columns] + [
    [clean_sheet_value(value) for value in row]
    for row in results_export.to_numpy()
]

try:
    results_worksheet = spreadsheet.worksheet(RESULTS_SHEET_NAME)
except gspread.WorksheetNotFound:
    results_worksheet = spreadsheet.add_worksheet(
        title=RESULTS_SHEET_NAME,
        rows=max(len(result_values), 2),
        cols=len(result_columns)
    )

results_worksheet.resize(
    rows=max(len(result_values), 2),
    cols=len(result_columns)
)
results_worksheet.clear()

result_end_cell = gspread.utils.rowcol_to_a1(
    len(result_values),
    len(result_columns)
)
results_worksheet.update(
    range_name=f"A1:{result_end_cell}",
    values=result_values,
    value_input_option="RAW"
)

# Formato visual; los datos ya quedan escritos aunque falle el formato.
try:
    results_worksheet.freeze(rows=1)
    last_column_letter = gspread.utils.rowcol_to_a1(
        1, len(result_columns)
    ).rstrip("1")
    results_worksheet.format(
        f"A1:{last_column_letter}1",
        {
            "backgroundColor": {"red": 0.12, "green": 0.28, "blue": 0.47},
            "textFormat": {
                "bold": True,
                "foregroundColor": {"red": 1, "green": 1, "blue": 1}
            }
        }
    )

    for column in ["ept_old", "ept_new", "ept_change_min"]:
        column_number = result_columns.index(column) + 1
        column_letter = gspread.utils.rowcol_to_a1(
            1, column_number
        ).rstrip("1")
        results_worksheet.format(
            f"{column_letter}2:{column_letter}{max(len(result_values), 2)}",
            {"numberFormat": {"type": "NUMBER", "pattern": "0.0"}}
        )

    change_pct_column_number = result_columns.index("ept_change_pct") + 1
    change_pct_column_letter = gspread.utils.rowcol_to_a1(
        1, change_pct_column_number
    ).rstrip("1")
    results_worksheet.format(
        f"{change_pct_column_letter}2:{change_pct_column_letter}{max(len(result_values), 2)}",
        {"numberFormat": {"type": "PERCENT", "pattern": "0%"}}
    )
except Exception as formatting_error:
    print(f"Advertencia de formato en results_export: {formatting_error}")

# Solo después de escribir results_export se marca el input como ejecutado.
if "last_executed_at" in input_headers_original:
    last_executed_at_col = input_headers_original.index("last_executed_at") + 1
    last_executed_values = [
        [row[last_executed_at_col - 1] if len(row) >= last_executed_at_col else ""]
        for row in input_values
    ]
    last_executed_values[0] = ["last_executed_at"]
else:
    last_executed_at_col = len(input_headers_original) + 1
    if last_executed_at_col > input_worksheet.col_count:
        input_worksheet.resize(
            rows=max(input_worksheet.row_count, len(input_values), 2),
            cols=last_executed_at_col
        )
    last_executed_values = [["last_executed_at"]] + [
        [""] for _ in input_values[1:]
    ]

for sheet_row_number in input_rows_to_mark:
    while len(last_executed_values) < sheet_row_number:
        last_executed_values.append([""])
    last_executed_values[sheet_row_number - 1] = [executed_at]

last_executed_at_column_letter = gspread.utils.rowcol_to_a1(
    1, last_executed_at_col
).rstrip("1")
input_worksheet.update(
    range_name=(
        f"{last_executed_at_column_letter}1:"
        f"{last_executed_at_column_letter}{len(last_executed_values)}"
    ),
    values=last_executed_values,
    value_input_option="RAW"
)

print(
    f"Hoja '{RESULTS_SHEET_NAME}' reemplazada con "
    f"{len(results_export):,} vendors exportados."
)
print(
    f"'{INPUT_SHEET_NAME}' se mantuvo intacta; "
    f"last_executed_at actualizado en {len(input_rows_to_mark):,} filas."
)
print("Origen del ajuste:")
display(
    export_audit.groupby(
        ["adjustment_origin", "adjustment_scope"],
        dropna=False
    ).size().rename("vendors").reset_index()
)
display(results_export.head(20))

Hoja 'results_export' reemplazada con 7 vendors exportados.
'to_adjust_now' se mantuvo intacta; last_executed_at actualizado en 11 filas.
Origen del ajuste:


TypeError: 'module' object is not callable